In [ ]:
# define Meta data

# rec_loops = 3



### Paper : Less is More: Recursive Reasoning with Tiny Networks, Oct 2025
(https://arxiv.org/pdf/2510.04871)

## Problem Overview

We study Recursive Reasoning using Tiny Networks (TRM).

Goal:
Instead of increasing model size, we improve reasoning by:
- reusing the same network multiple times
- learning when to halt (q_halt)

Key Idea:
Compute multiple reasoning steps and decide dynamically when to stop.

In [ ]:
import torch
from torch import nn

## Model Architecture

Components:
- f: prediction head
- q_halt: halting probability

At each step t:
1. Model updates hidden state
2. Predicts output
3. Predicts whether to stop

Final output depends on halting decision.

- Latent layers are not directly tracked in gradients.
- They are used to refine the internal states.
- This helps y and z move toward a stable representation before reasoning begins.

In [ ]:
class TinyRModel(nn.Module):
  def __init__(self, hidden_size, vocab_size):
    super().__init__()
    self.net = nn.Sequential(nn.Linear(hidden_size, hidden_size),
                              nn.ReLU(),
                              nn.Linear(hidden_size, hidden_size))

    self.embedding = nn.Embedding(vocab_size, hidden_size)
    self.out = nn.Linear(hidden_size, vocab_size)
    self.halt = nn.Linear(hidden_size, 1)



  def latent(self, x, y, z, n=6):
      # Phase 1: update z (n times), x included
      for i in range(n):
          z = self.net(x + y + z)   # element-wise add → D-dim input

      # Phase 2: update y once, WITHOUT x
      y = self.net(y + z)           # same net, different input mix

      return y, z



  def forward(self, x, y, z, T, n=6):

    with torch.no_grad():
      for j in range(T-1):
        y, z= self.latent(x,y,z,n)

    y, z = self.latent(x, y, z, n)

    f_out = self.out(y)

    halt= self.halt(y) #calculate halt from y

    return f_out, halt, y.detach(), z.detach()

### Dataset Description: Majority Task
The dataset consists of fixed-length binary sequences ($0$s and $1$s). The goal is to classify the sequence based on the majority bit:
- **Label 1**: If the count of $1$s is greater than the count of $0$s.
- **Label 0**: If the count of $0$s is greater than the count of $1$s.

Ties are excluded to ensure unambiguous labeling. This serves as a simple reasoning task for the recursive model.

In [ ]:
# Simple dataset

import torch
import random
from torch.utils.data import Dataset, DataLoader, random_split

# ── config ───────────────────────────────────────────────────────────────
hidden_size = 64
batch_size  = 32
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
vocab_size  = 2    # only real tokens: 0 and 1
num_classes = 2    # label 0 or 1
seq_len     = 15   # fixed length — no padding needed

# ── dataset ──────────────────────────────────────────────────────────────
class MajorityDataset(Dataset):
    """
    Fixed-length binary sequences, no padding, no masking.
    Label = 1 if 1s are majority, 0 if 0s are majority.
    Ties skipped to avoid ambiguous labels.
    """
    def __init__(self, num_samples=10000, seq_len=seq_len):
        self.samples = self._generate(num_samples, seq_len)

    def _generate(self, n, seq_len):
        samples = []
        while len(samples) < n:
            seq   = [random.randint(0, 1) for _ in range(seq_len)]
            ones  = sum(seq)
            zeros = seq_len - ones
            if ones == zeros:       # skip ties
                continue
            label = 1 if ones > zeros else 0
            samples.append((seq, label))
        return samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, label = self.samples[idx]
        return {
            "input_ids": torch.tensor(seq,   dtype=torch.long),
            "labels":    torch.tensor(label, dtype=torch.long),
        }

# ── dataloaders ──────────────────────────────────────────────────────────
dataset  = MajorityDataset(num_samples=10000)
val_size = 1000
train_set, val_set = random_split(dataset, [len(dataset) - val_size, val_size])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)

## Loss Function

We use two losses:

1. Prediction Loss:
   Cross entropy between prediction and label

2. Halting Loss:
   Binary classification:
   - 1 if prediction correct
   - 0 otherwise

This encourages:
→ stop when correct
→ continue when wrong

In [ ]:
def Q_LOSS(q_halt, f_out, y_true):
    target = (f_out.argmax(1) == y_true).float().unsqueeze(1)
    return F.binary_cross_entropy_with_logits(q_halt, target)

In [ ]:
import torch
import torch.nn.functional as F



# --- Training Loop ---

def model_training_and_validation_with_mask(c, T, n_sup):
  model = TinyRModel(hidden_size, vocab_size).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

  criterion = nn.CrossEntropyLoss(reduction='none')

  running_loss = 0
  running_q = 0
  running_halt = 0

  print('Training Logs ------ \n')
  model.train()
  for step, batch in enumerate(train_loader):
      input_ids = batch['input_ids'].to(device)
      labels = batch['labels'].to(device)

      active_mask = torch.ones(input_ids.size(0), dtype=torch.bool, device=device)

      x = model.embedding(input_ids).mean(dim=1)

      # Initial states
      y = torch.zeros(input_ids.size(0), hidden_size, device=device)
      z = torch.zeros(input_ids.size(0), hidden_size, device=device)

      step_loss = step_q = 0
      overall_loss = torch.tensor(0.0, device=device)

      for i in range(n_sup):

          f_out, halt, y_new, z_new = model(x, y, z, T, n=6)

          mask_expand = active_mask.unsqueeze(1)

          y = torch.where(mask_expand, y_new.detach(), y)
          z = torch.where(mask_expand, z_new.detach(), z)

          loss = criterion(f_out, labels)
          q_loss = Q_LOSS(halt, f_out, labels)

          #ONLY keep loss for active samples
          mask = active_mask.float().unsqueeze(1)
          num_active = mask.sum() + 1e-8

          overall_loss+= ((loss.unsqueeze(1) + c * q_loss) * mask).sum()/num_active

          step_loss += (loss.squeeze() * active_mask).sum().item() / (active_mask.sum().item() + 1e-8)
          step_q += (q_loss.squeeze() * active_mask).sum().item() / (active_mask.sum().item() + 1e-8)

          # per-sample halting
          halt_decision = (halt.squeeze() > 0) & active_mask

          # deactivate halted samples
          active_mask = active_mask & (~halt_decision)

          if active_mask.sum() == 0:
              break

      # ONE update

      n_taken = i + 1

      overall_loss /= n_taken

      optimizer.zero_grad()
      overall_loss.backward()
      optimizer.step()

      running_loss += step_loss / n_taken
      running_q += step_q / n_taken

      if step % 50 == 0 and step > 0:
          print(f"Step {step} | CE: {running_loss/50:.4f} | Q-Loss: {running_q/50:.4f} | n_taken: {n_taken}")
          running_loss = running_q = 0

  # --- Validation Loop ---

  print('\n\nValidation Logs ------ \n')
  model.eval()
  correct = 0
  total = 0

  val_running_loss = val_running_q = 0

  with torch.no_grad():
      for val_step, batch in enumerate(val_loader):
          input_ids = batch['input_ids'].to(device)
          labels = batch['labels'].to(device)

          x = model.embedding(input_ids).mean(dim=1)
          y = torch.zeros(input_ids.size(0), hidden_size, device=device)
          z = torch.zeros(input_ids.size(0), hidden_size, device=device)

          active_mask = torch.ones(input_ids.size(0), dtype=torch.bool, device=device)

          batch_val_loss = 0
          batch_val_halt = 0
          batch_val_q = 0

          overall_val_loss = torch.tensor(0.0, device=device)

          for i in range(n_sup):
              f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)

              mask_expand = active_mask.unsqueeze(1)

              y = torch.where(mask_expand, y_new.detach(), y)
              z = torch.where(mask_expand, z_new.detach(), z)


              val_loss = criterion(f_out_val, labels)
              val_q_loss = Q_LOSS(val_halt, f_out_val, labels)

              #ONLY keep loss for active samples
              mask = active_mask.float().unsqueeze(1)
              num_active = mask.sum() + 1e-8

              overall_val_loss+= ((val_loss.unsqueeze(1) + c * val_q_loss) * mask).sum()/num_active

              batch_val_loss += (val_loss.squeeze() * active_mask).sum().item() / (active_mask.sum().item() + 1e-8)
              batch_val_q += (val_q_loss.squeeze() * active_mask).sum().item() / (active_mask.sum().item() + 1e-8)

              # per-sample halting
              val_halt_decision = (val_halt.squeeze() > 0) & active_mask

              active_mask = active_mask & (~val_halt_decision)

              if active_mask.sum() == 0:
                  break

          n_taken_val = i + 1

          overall_val_loss /= n_taken_val

          val_running_loss += batch_val_loss / n_taken_val
          val_running_q += batch_val_q / n_taken_val

          if val_step % 10 == 0 and val_step > 0:
              print(f"Val Step {val_step} | CE: {val_running_loss/10:.4f} | Q-Loss: {val_running_q/10:.4f} | n_taken: {n_taken_val}")
              val_running_loss = val_running_q = val_running_halt = 0

          predictions = f_out_val.argmax(dim=1)
          correct += (predictions == labels).sum().item()
          total += labels.size(0)

  print(f"\nValidation Accuracy: {100 * correct / total:.2f}%")

In [ ]:
import torch
import torch.nn.functional as F




def model_training_and_validation(c, T, n_sup):

    running_loss = 0
    running_q = 0
    running_halt = 0

    criterion = nn.CrossEntropyLoss()

    model = TinyRModel(hidden_size, vocab_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


    # --- Training Loop ---
    print('Training Logs ------ \n')

    model.train()
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)

        x = model.embedding(input_ids).mean(dim=1)

        # Initial states
        y = torch.zeros(input_ids.size(0), hidden_size, device=device)
        z = torch.zeros(input_ids.size(0), hidden_size, device=device)

        step_loss = step_q =  0
        overall_loss = 0

        for i in range(n_sup):

            f_out, halt, y_new, z_new= model(x, y, z, T, n=6)

            # Update states for next supervision step
            y, z = y_new.detach(), z_new.detach()

            loss = criterion(f_out, labels)
            q_loss_val = Q_LOSS(halt, f_out, labels)
            overall_loss += loss + c * q_loss_val

            step_loss += loss.item()
            step_q += q_loss_val.item()

            if (halt > 0).all():
                break

        n_taken = i + 1
        overall_loss /= n_taken

        optimizer.zero_grad()
        overall_loss.backward()
        optimizer.step()


        running_loss += step_loss / n_taken
        running_q += step_q / n_taken

        if step % 50 == 0 and step > 0:
            print(f"Step {step} | CE: {running_loss/50:.4f} | Q-Loss: {running_q/50:.4f} | n_taken: {n_taken}")
            running_loss = running_q = 0

    # --- Validation Loop ---

    print('\n\nValidation Logs ------ \n')
    # Moving validation outside the training step loop for efficiency
    model.eval()
    correct = 0
    total = 0
    val_running_loss = val_running_q = 0

    with torch.no_grad():
        for val_step, batch in enumerate(val_loader):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            x = model.embedding(input_ids).mean(dim=1)
            y = torch.zeros(input_ids.size(0), hidden_size, device=device)
            z = torch.zeros(input_ids.size(0), hidden_size, device=device)

            batch_val_loss = 0
            batch_val_q = 0

            for i in range(n_sup):
                f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)
                y, z = y_new.detach(), z_new.detach()

                v_loss = criterion(f_out_val, labels)
                v_q_loss = Q_LOSS(val_halt, f_out_val, labels)

                batch_val_loss += v_loss.item()
                batch_val_q += v_q_loss.item()

                if (val_halt > 0).all():
                    break

            n_taken_val = i + 1
            val_running_loss += batch_val_loss / n_taken_val
            val_running_q += batch_val_q / n_taken_val

            if val_step % 10 == 0 and val_step > 0:
                print(f"Step {val_step} | CE: {val_running_loss/10:.4f} | Q-Loss: {val_running_q/10:.4f} | n_taken: {n_taken_val}")
                val_running_loss = val_running_q = 0

            predictions = f_out_val.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    print(f"\nValidation Accuracy: {100 * correct / total:.2f}%")

In [ ]:
model_training_and_validation(c=0.01, T=3, n_sup=16)

Training Logs ------ 

Step 50 | CE: 0.6958 | Q-Loss: 0.7067 | n_taken: 1
Step 100 | CE: 0.6449 | Q-Loss: 0.6593 | n_taken: 1
Step 150 | CE: 0.3737 | Q-Loss: 0.5800 | n_taken: 1
Step 200 | CE: 0.0107 | Q-Loss: 0.3715 | n_taken: 1
Step 250 | CE: 0.0005 | Q-Loss: 0.1600 | n_taken: 1


Validation Logs ------ 

Step 10 | CE: 0.0002 | Q-Loss: 0.0755 | n_taken: 1
Step 20 | CE: 0.0002 | Q-Loss: 0.0662 | n_taken: 1
Step 30 | CE: 0.0002 | Q-Loss: 0.0645 | n_taken: 1

Validation Accuracy: 100.00%


In [ ]:
model_training_and_validation_with_mask(c=0.01, T=3, n_sup=16)

Training Logs ------ 

Step 50 | CE: 0.6978 | Q-Loss: 0.7050 | n_taken: 16
Step 100 | CE: 0.6272 | Q-Loss: 0.7003 | n_taken: 16
Step 150 | CE: 0.3909 | Q-Loss: 0.6587 | n_taken: 1
Step 200 | CE: 0.0300 | Q-Loss: 0.4358 | n_taken: 1
Step 250 | CE: 0.0013 | Q-Loss: 0.2042 | n_taken: 1


Validation Logs ------ 

Val Step 10 | CE: 0.0003 | Q-Loss: 0.0973 | n_taken: 1
Val Step 20 | CE: 0.0003 | Q-Loss: 0.0868 | n_taken: 1
Val Step 30 | CE: 0.0002 | Q-Loss: 0.0859 | n_taken: 1

Validation Accuracy: 100.00%


## Key Observations

- Using mean loss across steps harms per-example optimization
- If q_halt becomes negative early, future optimization is skipped
- Masking improves stability by preserving active samples


### Key Observation: Effect of Halting Coefficient (c)

- c controls the trade-off between prediction and halting behavior
- With small c (0.01), model halts after 1 step (n_taken = 1)
- This reduces effective reasoning depth despite high accuracy
- Indicates that halting must be carefully balanced to enable multi-step reasoning